In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tqdm.notebook as tqdm
from pathlib import Path
from pupil_apriltags import Detector

In [ ]:
folder = Path('/Users/gerry/DropboxGatech/Painting/Graffiti_media+data/2024-02-03_iros2024_homography_data/')
video_file = folder / 'C.mov'

In [ ]:
detector = Detector(families="tagStandard41h12",
                    nthreads=1,
                    quad_decimate=1.0,
                    quad_sigma=0.0,
                    refine_edges=1,
                    decode_sharpening=0.25,
                    debug=0)

In [ ]:
# Test
# Read first frame from video
cap = cv2.VideoCapture((video_file).as_posix())
success, image = cap.read()
assert success, "Error reading video"
plt.imshow(image)
cap.release()

In [ ]:
# Apriltag detection
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
detections = detector.detect(gray)
print(f'Found {len(detections)} tags')
for detection in detections:
    print(detection.tag_id, detection.tag_family, detection.corners, detection.center)

In [ ]:
# Now read more frames from video
EVERY_NTH_FRAME = 10
all_detections = {}

cap = cv2.VideoCapture((video_file).as_posix())
total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
with tqdm.tqdm(total=total_frames) as pbar:
    for i in range(int(total_frames)):
        success, image = cap.read()
        if not success:
            print(f'Error reading frame {i}')
            break
        if i % EVERY_NTH_FRAME == 0:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            detections = detector.detect(gray)
            all_detections[i] = detections
            pbar.set_description(f'Frame {i}: Found {len(detections)} tags')
        if i % 1000 == 0:
            # Save intermediate results
            name = video_file.name
            np.savez((folder / 'data' / f'{video_file.stem}_detections_every{EVERY_NTH_FRAME}.npz').as_posix(), all_detections=all_detections)
        pbar.update(1)
cap.release()
np.savez((folder / 'data' / f'{video_file.stem}_detections_every{EVERY_NTH_FRAME}.npz').as_posix(), all_detections=all_detections)

In [ ]:
with np.load(folder / 'data' / f'{video_file.stem}_detections_every{EVERY_NTH_FRAME}.npz', allow_pickle=True) as f:
    all_detections_ = f['all_detections'].item()
    print(max(all_detections_.keys()))